<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/v3/llm-inference/continuous-batching/continuous-batching.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("continuous-batching", ...)`


# Question 19: Implement Continuous Batching for LLM Inference

### Difficulty: Hard

### Problem Statement

In naive LLM serving, a batch of requests is processed together and the entire batch must wait until the longest sequence finishes. This wastes compute on short requests that finish early.

**Continuous batching** (also called **iteration-level scheduling**) solves this by allowing the scheduler to:
- **Evict** completed sequences from the batch immediately (when they hit EOS or max length).
- **Insert** new waiting requests into the freed slots without waiting for the whole batch to finish.

Every iteration runs **one batched forward pass** over all requests currently in the batch, so short requests stop costing compute the moment they finish and new requests start without waiting for the longest one. This is the core scheduling idea behind systems like vLLM, TGI, and Orca (Yu et al., 2022).

Your task is to:
1. Implement a `Request` class to track each inference request's state.
2. Implement a `ContinuousBatchScheduler` that manages an active batch, inserting and evicting requests dynamically.
3. Show that batching never changes a request's output, and that continuous batching achieves higher throughput than naive sequential processing.

---

### Requirements

1. **`Request`** class:
   - Stores `request_id`, `input_ids`, `generated_ids`, `max_gen_len`, `is_done` flag.
   - Tracks when the request was created and when it finished.

2. **`ContinuousBatchScheduler`** class:
   - `ContinuousBatchScheduler(max_batch_size, eos_token_id, vocab_size, greedy=False)`; `greedy=True` takes the argmax instead of sampling, which makes outputs reproducible and testable.
   - `add_request(request)`: Add a request to the waiting queue.
   - `step(model)`: Run one generation step:
     a. Fill empty batch slots from the waiting queue (FIFO, so no request is starved).
     b. Run **one** model forward pass for all active requests: right-pad their sequences (prompt + generated so far) to the longest with `eos_token_id`, and read each request's next-token logits at its **own last real position** (`length - 1`), not at the last column. Right padding is safe for a causal model — real positions never attend to the pads after them; left padding would need an attention mask and shifted position ids.
     c. Append generated tokens to each request.
     d. Evict completed requests (EOS token or max length reached) — their slots are refilled at the next step.
   - `run(model)`: Run until all requests (queued + active) are finished.

3. **Validation**: Submit 10 requests of varying lengths. With greedy decoding, check that every request produces exactly the tokens it produces when decoded alone. Compare throughput (tokens/sec and forward passes) of continuous batching vs. sequential processing, and count how many forward passes static batching would have needed.

---

### Constraints

- Use only PyTorch operations.
- Use a simple dummy model (e.g., random logit generation) so the focus is on the scheduling logic.
- Track timestamps for each request to show latency improvements.

> To keep the focus on scheduling, this scheduler recomputes every request's full prefix at each step. Production engines keep a per-request KV cache (paged in vLLM), mix the prefill of newly admitted requests with the decode steps of running ones in the same iteration, and preempt requests when KV-cache memory runs out.

---

### Company Tags

Perplexity, Together AI, Anyscale, Meta

---

<details>
  <summary>Hint</summary>

  - The scheduler maintains two data structures: a `waiting_queue` (list of requests not yet started) and an `active_batch` (list of requests currently being generated).
  - At each `step()`, first fill empty slots in the active batch from the waiting queue, then run inference, then evict finished requests.
  - Batch the forward pass: right-pad every active sequence to the longest, call the model once on `(num_active, max_len)`, and gather `logits[i, length_i - 1]`. Calling the model once per request is *not* batching — it is sequential decoding with extra bookkeeping.
  - The dummy model here is causal and context-dependent (each position sees a running average of the prefix), so a padding mistake changes the generated tokens — that is what the greedy equivalence test catches.

</details>

---

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from dataclasses import dataclass, field
from typing import List, Optional

In [ ]:
# Configuration
torch.manual_seed(42)

vocab_size = 256
d_model = 128
max_batch_size = 4
eos_token_id = 0  # Token ID 0 is EOS

device = "cpu"
print(f"Vocab size: {vocab_size}, Max batch size: {max_batch_size}, EOS token: {eos_token_id}")

In [ ]:
class DummyLLM(nn.Module):
    """
    A tiny causal "language model" for testing the scheduler.
    The logits at position t depend on token t and on the mean embedding of tokens
    0..t, so -- like a real transformer -- they depend on the context but never on
    later positions. The model itself is not the focus -- the scheduling logic is.
    """
    
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.fc = nn.Linear(d_model, vocab_size)
    
    def forward(self, input_ids):
        """input_ids: (batch_size, seq_len) -> logits: (batch_size, seq_len, vocab_size)"""
        x = self.embedding(input_ids)                                        # (B, S, d_model)
        counts = torch.arange(1, x.shape[1] + 1, device=x.device).view(1, -1, 1)
        prefix_mean = x.cumsum(dim=1) / counts                               # causal context
        return self.fc(x + prefix_mean)


print("DummyLLM defined.")

In [ ]:
@dataclass
class Request:
    """
    Represents a single inference request.
    
    Fields:
        request_id: Unique identifier
        input_ids: List of input token IDs (the prompt)
        max_gen_len: Maximum number of tokens to generate
        generated_ids: List of generated token IDs (grows over time)
        is_done: Whether generation is complete
        start_time: When the request started processing
        end_time: When the request finished
    """
    # TODO: Define the fields listed above
    # Use field(default_factory=list) for generated_ids
    # Use None as default for start_time and end_time
    # Use False as default for is_done
    ...
    
    def get_full_sequence(self):
        """Return the full token sequence (input + generated)."""
        # TODO
        ...
    
    def total_len(self):
        """Return the total sequence length."""
        # TODO
        ...


class ContinuousBatchScheduler:
    """
    Continuous batching scheduler that dynamically adds/removes requests.
    
    - Maintains a waiting queue and an active batch.
    - At each step, fills empty slots, runs inference, evicts finished requests.
    """
    
    def __init__(self, max_batch_size, eos_token_id, vocab_size, greedy=False, device="cpu"):
        # TODO: Initialize waiting_queue, active_batch, completed list, step_count,
        #       and store max_batch_size, eos_token_id, vocab_size, greedy, device
        ...
    
    def add_request(self, request):
        """Add a request to the waiting queue."""
        # TODO
        ...
    
    def _fill_batch(self):
        """Move requests from waiting queue to active batch (up to max_batch_size)."""
        # TODO: While there are empty slots and waiting requests,
        #       pop from waiting queue, set start_time, add to active batch
        ...
    
    def _evict_completed(self):
        """Remove completed requests from the active batch."""
        # TODO: Move done requests to completed list, set end_time,
        #       keep only active (not done) requests
        ...
    
    def step(self, model):
        """
        Run one generation step:
        1. Fill empty batch slots from waiting queue.
        2. Run ONE forward pass over all active requests and pick each next token
           (argmax if self.greedy, else sample).
        3. Check for completion (EOS or max length).
        4. Evict completed requests.
        
        Returns:
            bool: True if there are still active or waiting requests.
        """
        # TODO: Implement the step logic described above.
        # - Right-pad every active sequence (prompt + generated) to the longest one
        #   with eos_token_id -> input_ids of shape (num_active, max_len)
        # - logits = model(input_ids); for request i use logits[i, length_i - 1]
        #   (its last REAL token, not the last column, which may be padding)
        ...
    
    def run(self, model):
        """
        Run the scheduler until all requests are completed.
        Returns the list of completed requests.
        """
        # TODO: Call step() in a loop until no more work to do
        ...


print("Request and ContinuousBatchScheduler classes defined. Fill in the TODO sections above.")

In [ ]:
# ============================================================
# Validation: Continuous Batching vs Sequential Processing
# ============================================================

torch.manual_seed(42)
model = DummyLLM(vocab_size, d_model).to(device).eval()

# Create 10 requests with varying prompt lengths and max generation lengths
requests = []
for i in range(10):
    prompt_len = torch.randint(3, 10, (1,)).item()
    max_gen_len = torch.randint(5, 20, (1,)).item()
    input_ids = torch.randint(1, vocab_size, (prompt_len,)).tolist()  # Avoid EOS=0 in prompt
    requests.append(Request(
        request_id=i,
        input_ids=input_ids,
        max_gen_len=max_gen_len
    ))
    print(f"Request {i}: prompt_len={prompt_len}, max_gen_len={max_gen_len}")

print(f"\nTotal requests: {len(requests)}")
print(f"Max batch size: {max_batch_size}")
print()

# Greedy decoding everywhere, so that outputs can be compared token for token.

# --- Method 1: Sequential processing (one request at a time, one forward pass per token) ---
import copy
seq_requests = copy.deepcopy(requests)

start = time.time()
total_tokens_seq = 0
with torch.no_grad():
    for req in seq_requests:
        req.start_time = time.time()
        current_ids = torch.tensor([req.input_ids], device=device)
        for _ in range(req.max_gen_len):
            logits = model(current_ids)
            next_token = logits[0, -1].argmax().item()
            req.generated_ids.append(next_token)
            if next_token == eos_token_id:
                break
            current_ids = torch.cat([current_ids, torch.tensor([[next_token]], device=device)], dim=1)
        req.end_time = time.time()
        total_tokens_seq += len(req.generated_ids)
seq_time = time.time() - start

# --- Method 2: Continuous batching ---
cb_requests = copy.deepcopy(requests)
scheduler = ContinuousBatchScheduler(max_batch_size, eos_token_id, vocab_size, greedy=True)
for req in cb_requests:
    scheduler.add_request(req)

start = time.time()
with torch.no_grad():
    completed = scheduler.run(model)
cb_time = time.time() - start

total_tokens_cb = sum(len(r.generated_ids) for r in completed)

# --- Correctness: batching must not change any request's output ---
seq_by_id = {r.request_id: r.generated_ids for r in seq_requests}
for r in completed:
    assert r.generated_ids == seq_by_id[r.request_id], (
        f"Request {r.request_id}: continuous batching produced {r.generated_ids}, "
        f"decoding it alone produced {seq_by_id[r.request_id]} -- check padding / logit positions")
assert sorted(r.request_id for r in completed) == list(range(len(requests))), "every request must finish exactly once"
print("Correctness PASSED: every request decodes exactly as it does on its own.\n")

# Static batching for comparison: fixed groups of max_batch_size, each runs until its longest member is done
gen_lens = [len(seq_by_id[i]) for i in range(len(requests))]
static_steps = sum(max(gen_lens[i:i + max_batch_size]) for i in range(0, len(gen_lens), max_batch_size))

print("=" * 50)
print("Results")
print("=" * 50)
print(f"\nSequential: {seq_time:.4f}s, {total_tokens_seq} tokens, {total_tokens_seq/seq_time:.1f} tokens/sec, {total_tokens_seq} forward passes")
print(f"Continuous: {cb_time:.4f}s, {total_tokens_cb} tokens, {total_tokens_cb/cb_time:.1f} tokens/sec, {scheduler.step_count} forward passes")
print(f"Static batching would need {static_steps} forward passes for the same work")
# The forward-pass count is the robust comparison. These runs take milliseconds on a toy
# CPU model, so the wall-clock ratio is noisy (it can even come out below 1x here); the
# win shows up when a batched forward costs about as much as a one-row one (memory-bound
# LLM decoding on a GPU).
print(f"\nWall-clock speedup: {seq_time / cb_time:.2f}x")
print()

# Show per-request latency for continuous batching
print("Per-request completion times (continuous batching):")
for r in sorted(completed, key=lambda x: x.request_id):
    latency = r.end_time - r.start_time
    print(f"  Request {r.request_id}: generated {len(r.generated_ids)} tokens, latency={latency:.4f}s")